# Outline

- Preparing Workspace

Importing packages, defining file paths, running user defined functions, setting API key, ...

- Preparing Imports

This section imports the "Census Configuration File.xlsx" and sets the user defined inputs to objects that are used throughout the script

- Importing
- Processing
- Exporting

***

## Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

## Preparing Imports

***

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying Census data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]

# View
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import about table
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
MOE_thresh = df_about['MOE Threshold'].values[0]
print(folder)
print('MOE threshold: ' + str(MOE_thresh) + '%')

In [ ]:
## Import Variable Mapping
df_inputs = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = import_tab)

# Set years
years_to_import = list(range(year_start, year_end+1))

## For DEC data
if estimate == 'DEC':

    # Reset years to import for DEC
    # Set DEC variables to import
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = estimate)
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    if margin_of_error == 'Yes':
        df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)

    years_to_import = [2000, 2010, 2020]
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = ['NAME'] + unique(df_vars[df_vars['Year'] == year]['ID'].to_list())
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
    
    # view
    print(dict_fips)
    print(dict_vars)

## For ACS1 or ACS5 data
if sample_type in ['ACS', 'SUBJECT']:

    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    # Remove 2020 if pulling ACS1 tables (Census did not take an ACS1 sample in 2020)
    # Set tables and variables to import    

    if estimate == 'ACS1':
        try:
            years_to_import.remove(2020)
        except Exception as e: print(e)

    if margin_of_error == 'Yes':
        df_vars['ID_Attributes'] = df_vars['ID_Attributes'].apply(ME_split)
        list_vars = ['NAME'] + df_vars['ID_Attributes'].to_list()
    else:
        list_vars = ['NAME'] + df_vars['ID'].to_list()

    if sample_type == 'ACS':
        tables = df_vars['Table'].unique()
        print(tables)

    # For tract and county level pull
    if import_tab == 'Counties':
        
        # Import County FIPS mapping
        # Convert to dictionary object for easy state-county combination importing
        df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                , sheet_name = 'CountyFIPS'
                                , dtype = {'State FIPS': object, 'County FIPS': object})
        df_fips = df_fips[
                        (df_fips['State'].isin(df_inputs['states'].values))
                        & (df_fips['County Name'].isin(df_inputs['counties'].values))
        ]
        dict_fips = df_fips.copy()
        dict_fips = dict_fips[['State FIPS', 'County FIPS']]
        dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
        
        for key in list(dict_fips.keys()):
            dict_fips[key] = ",".join(dict_fips[key])
    
        # view
        print(dict_fips)
        print(list_vars)

    
    # For MSA level pull
    if import_tab == 'MSA':
    
        # Set MSAs to import
        df_inputs['msa'] = df_inputs['msa'].astype("string")
        msa_to_import = df_inputs['msa'].values
        msa_to_import = ",".join(msa_to_import)
    
        # view
        print(tables)
        print(msa_to_import)
        print(list_vars)


## For PUMS data
if sample_type == 'PUMS':

    # Remove 2012-2015 if pulling PUMS tables (they only reported at the state level for PUMS on these years)
    # Create dictionary of variable mappings by year (sometimes the variable name changes over time)
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    if 'H' in df_vars['Table Type'].unique():
        table_type = 'H'
    else:
        table_type = 'P'
    print('PUMS table roll up: ' + table_type)

    if margin_of_error == 'Yes':
        df_vars.loc[(df_vars['ID'].str.contains('WGTP')) & (df_vars['Table Type'] == table_type), 'Include'] = 'Yes'
    
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    if table_type == 'P':
        weight = 'PWGTP'
    if table_type == 'H':
        weight = 'WGTP'
    groups  = list(df_vars[df_vars['Data Type'].str.contains('group')]['ID2'].unique())
    groups2 = list(df_vars[df_vars['Data Type'] == 'group']['ID2'].unique())
    print(groups)
    print(groups2)
    
    if estimate == 'ACS5':
        try:
            years_to_import.remove(2012)
            years_to_import.remove(2013)
            years_to_import.remove(2014)
            years_to_import.remove(2015)
        except Exception as e: print(e)

    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
            
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[(df_vars['Year'] == year) & (df_vars['Data Type'].str.contains('group'))]['ID'].to_list()) + unique(df_vars[(df_vars['Year'] == year) & (df_vars['Data Type'] == 'integer')]['ID'].to_list()) + [weight]
        
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    df_fips_pums = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                                 , sheet_name = 'PUMAcodes'
                                 , dtype = {'STATEFP': object, 'COUNTYFP': object, 'TRACTCE': object, 'PUMA5CE': object})
    df_fips_pums = df_fips_pums.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS'})

    df_fips = df_fips.merge(df_fips_pums[['State FIPS', 'County FIPS', 'PUMA5CE']].drop_duplicates(), on = ['State FIPS', 'County FIPS'])
    df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)]
    dict_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = dict_fips[['State FIPS', 'PUMA5CE']].drop_duplicates()
    
    dict_fips = dict_fips.groupby('State FIPS')['PUMA5CE'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])

    # view
    print(dict_fips)
    print(dict_vars)



if estimate == 'CPS':
    
    df_vars = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = sample_type)
    df_vars = df_vars[df_vars['Year'].isin(years_to_import)]
    df_vars = df_vars[df_vars['Indicator Name'].str.contains(indicator_name).replace(np.nan, False)]
    df_vars = df_vars[df_vars['Include'] == 'Yes']
    
    dict_vars = {}
    for year in years_to_import:
        dict_vars[str(year)] = unique(df_vars[df_vars['Year'] == year]['ID'].to_list()) + [df_vars[df_vars['Year'] == year]['Suggested Weight'].values[0]]
    
    # Import County FIPS mapping
    # Convert to dictionary object for easy state-county combination importing
    df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
    
    df_fips = df_fips[
                    (df_fips['State'].isin(df_inputs['states'].values))
                    & (df_fips['County Name'].isin(df_inputs['counties'].values))
    ]
    dict_fips = df_fips.copy()
    dict_fips = dict_fips[['State FIPS', 'County FIPS']]
    dict_fips = dict_fips.groupby('State FIPS')['County FIPS'].apply(list).to_dict()
    
    for key in list(dict_fips.keys()):
        dict_fips[key] = ",".join(dict_fips[key])
        
    # view
    print(dict_fips)
    print(dict_vars)


# view
df_vars.head(3)

***

## Importing

***

In [ ]:
start_time = time.time()

# Import Census Bureau data to url mapping
df_urls = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'URL')

## For ACS tables
if sample_type == 'ACS':
  
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling ACS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # reduce all tables/variables pulled into one table

    list_df_census = []
    
    for table in tables:
    
        print("")
        print("Table ID: " + table)
        print("")
        list_df_tables = []
    
        df_table = df_vars[df_vars['Table'] == table]
        if margin_of_error == 'Yes':
            list_table_vars = [['NAME'] + df_table['ID_Attributes'].to_list()[x:x+20] for x in range(0, len(df_table['ID_Attributes'].to_list()), 20)]
        else:
            list_table_vars = [['NAME'] + df_table['ID'].to_list()[x:x+45] for x in range(0, len(df_table['ID'].to_list()), 45)]
        
        list_variables = []
        for x in list_table_vars:
            list_variables.append(",".join(x))
        
        list_df_vars = []
        
        for variables in list_variables:
            print("Variables: " + variables)
            list_df_years = []

            if import_tab == 'Counties':
                for state in list(dict_fips.keys()):
                    print('State: ' + state)
                    for year in tqdm(years_to_import):
                        try:
                            list_df_years.append(
                                query_census(df_urls      = df_urls
                                              , api_key   = api_key
                                              , estimate  = estimate
                                              , sample    = sample_type
                                              , geography = geography
                                              , variables = variables
                                              , year      = year
                                              , state     = state
                                              , county    = dict_fips[state])
                            )
                        except Exception as e: print(e)
                df_years = pd.concat(list_df_years)
                            
            if import_tab == 'MSA':
                for year in tqdm(years_to_import):
                    try:
                        list_df_years.append(
                            query_census(df_urls      = df_urls
                                          , api_key   = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , msa       = msa_to_import)
                        )
                    except Exception as e: print(e)
                df_years = pd.concat(list_df_years)

            list_df_vars.append(df_years)
        
        if geography == 'Tracts':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'Counties':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year'], how = 'outer'), list_df_vars)
        if geography == 'MSA':
            df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'outer'), list_df_vars)

        list_df_census.append(df_vars_all)
        print("All variables from table ID " + table + " have been reduced together into one table")
        print("")

    print("")
    print("Reducing all tables together into one final table...")
    print("")
    
    if geography == 'Tracts':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()
    if geography == 'MSA':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']), list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']).reset_index()



## For SUBJECT tables
if sample_type == 'SUBJECT':
    ## Import data and concatenate onto ID fields ##
    print("Importing and compiling ACS Subject data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties (or MSAs)
    # reduce all tables/variables pulled into one table

    list_df_census = []
    if margin_of_error == 'Yes':
        list_table_vars = [['NAME'] + df_vars['ID_Attributes'].to_list()[x:x+20] for x in range(0, len(df_vars['ID_Attributes'].to_list()), 20)]
    else:
        list_table_vars = [['NAME'] + df_vars['ID'].to_list()[x:x+45] for x in range(0, len(df_vars['ID'].to_list()), 45)]
    
    list_variables = []
    for x in list_table_vars:
        list_variables.append(",".join(x))
    
    list_df_vars = []
    
    for variables in list_variables:
        print("Variables: " + variables)
        list_df_years = []
        if import_tab == 'Counties':
            for state in list(dict_fips.keys()):
                print('State: ' + state)
                for year in tqdm(years_to_import):
                    try:
                        list_df_years.append(
                            query_census(df_urls      = df_urls
                                          , api_key   = api_key
                                          , estimate  = estimate
                                          , sample    = sample_type
                                          , geography = geography
                                          , variables = variables
                                          , year      = year
                                          , state     = state
                                          , county    = dict_fips[state])
                        )
                    except Exception as e: print(e)
            df_years = pd.concat(list_df_years)
                        
        if import_tab == 'MSA':
            for year in tqdm(years_to_import):
                try:
                    list_df_years.append(
                        query_census(df_urls      = df_urls
                                      , api_key   = api_key
                                      , estimate  = estimate
                                      , sample    = sample_type
                                      , geography = geography
                                      , variables = variables
                                      , year      = year
                                      , msa       = msa_to_import)
                    )
                except Exception as e: print(e)
            df_years = pd.concat(list_df_years)
        list_df_vars.append(df_years)
    
    if geography == 'Tracts':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year'], how = 'outer'), list_df_vars)
    if geography == 'Counties':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year'], how = 'outer'), list_df_vars)
    if geography == 'MSA':
        df_vars_all = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year'], how = 'outer'), list_df_vars)
    list_df_census.append(df_vars_all)

    print("")
    print("Reducing all tables together into one final table...")
    print("")
    
    if geography == 'Tracts':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'tract', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'state', 'county', 'Year']), list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()
    if geography == 'MSA':
        df_census_raw = ft.reduce(lambda left, right: pd.merge(left, right, on = ['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']), list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'metropolitan statistical area/micropolitan statistical area', 'Year']).reset_index()



## For DEC tables
if estimate == 'DEC':

    print("Importing and compiling Decennial data from the Census Bureau...")
    print("")

    # initialize empty list to store data frames
    # import multiple years and counties
    # iterate through each table and import all variables needed from each table
    # combine all years and counties
    # reduce all tables/variables pulled into one table
    
    list_df_census = []
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                list_df_census.append(
                    query_census(df_urls       = df_urls
                                   , api_key   = api_key
                                   , estimate  = estimate
                                   , sample    = sample_type
                                   , geography = geography
                                   , variables = ','.join(dict_vars[str(year)])
                                   , year      = year
                                   , state     = state
                                   , county    = dict_fips[state])
                )
            except Exception as e: print(e)
                    
    if geography == 'Tracts':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'tract', 'Year']).reset_index()
    if geography == 'Counties':
        df_census_raw = pd.concat(list_df_census)
        df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                              , left_on = ['state', 'county']
                                              , right_on = ['State FIPS', 'County FIPS'])
        df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
        df_census_raw = df_census_raw.set_index(['NAME', 'state', 'county', 'County Name', 'Year']).reset_index()


## For PUMS tables
if geography == 'PUMA':
    
    print("Importing and compiling PUMS data from the Census Bureau...")
    print("")
    
    # initialize empty list to store data frames
    # import multiple years and PUMAs
    # import all variables
    # combine all years and PUMAs
    # outer join variables onto ID fields for each geography type
    # calculate margin of error using replicate weights
    
    for state in list(dict_fips.keys()):
        print('State: ' + state)
        list_df_years = []
        for year in years_to_import:
            print("")
            print('Year: ' + str(year))
            list_df_vars = []
            try:
                list_table_vars = [dict_vars[str(year)][x:x+45] for x in range(0, len(dict_vars[str(year)]), 45)]
                
                list_variables = []
                for x in list_table_vars:
                    list_variables.append(",".join(x))

                print('Querying variables...')
                for variables in tqdm(list_variables):
                    df_pums = query_census(df_urls       = df_urls
                                             , api_key   = api_key
                                             , estimate  = estimate
                                             , sample    = sample_type
                                             , geography = geography
                                             , variables = 'PUMA,SERIALNO,SPORDER,' + variables
                                             , year      = year
                                             , state     = state
                                             , puma      = dict_fips[state])
                    df_pums['state'] = state
                    df_pums = df_pums.drop(['public use microdata area'], axis = 1)
                    list_df_vars.append(df_pums)

                df_vars_years = ft.reduce(lambda left, right: pd.merge(left, right, on = ['state', 'SERIALNO', 'Year', 'PUMA', 'SPORDER'], how = 'left'), list_df_vars)
                df_vars_years = df_vars_years.set_index(['state', 'SERIALNO', 'Year', 'PUMA', 'SPORDER']).reset_index()
                df_vars_years.columns = ['state', 'SERIALNO', 'Year', 'PUMA', 'SPORDER'] + dict_vars[str(np.max(years_to_import))]
                if (table_type == 'H') & ('SPORDER' in df_vars_years.columns):
                    df_vars_years = df_vars_years[df_vars_years['SPORDER'] == '1']
                df_vars_years = df_vars_years.drop('SPORDER', axis = 1)
                if margin_of_error == 'Yes':
                    print('Calculating margin of error using replicate weights...')
                    cols = [col for col in df_vars_years.columns if weight in col]
                    df_vars_years[cols] = df_vars_years[cols].astype(int)
                    cols_to_drop = cols[:-1]
                    cols = list(df_vars_years.drop(cols_to_drop, axis = 1).columns)
                    df_me = pd.melt(df_vars_years
                                     , id_vars    = cols
                                     , var_name   = 'replicates'
                                     , value_name = 'replicate_weights')
                    df_me['sq_diff'] = (df_me['replicate_weights'] - df_me[weight])**2
                    df_me = df_me.groupby(cols, as_index = False)['sq_diff'].agg(sum)
                    df_me['variance'] = df_me['sq_diff']*(4/80)
                    df_me['SE'] = np.sqrt(df_me['variance'])
                    df_me['ME'] = df_me['SE']*1.645
                    df_me = df_me[cols + ['ME']].drop_duplicates()
                    df_vars_years = df_vars_years.drop(cols_to_drop, axis = 1)
                    df_vars_years = df_vars_years.drop_duplicates()
                    df_vars_years = df_vars_years.merge(df_me, on = cols, how = 'left')
                    df_vars_years = df_vars_years.drop_duplicates()
                list_df_years.append(df_vars_years)
                
                print('Success!')
                
            except Exception as e: print(e)
                                
    df_census_raw = pd.concat(list_df_years)


## For CPS tables
if estimate == 'CPS':

    
    print("Importing and compiling CPS data from the Census Bureau...")
    print("")
    
    list_df_census = []

    for state in list(dict_fips.keys()):
        print('State: ' + state)
        for year in tqdm(years_to_import):
            try:
                temp = query_census(df_urls      = df_urls
                                     , api_key   = api_key
                                     , estimate  = estimate
                                     , sample    = sample_type
                                     , geography = geography
                                     , variables = 'HRHHID,HRHHID2,'+','.join(dict_vars[str(year)])
                                     , year      = year
                                     , state     = state
                                     , county    = dict_fips[state])
                list_df_census.append(temp)
            except Exception as e: print(e)
                
    df_census_raw = pd.concat(list_df_census)

    # merge county name onto table
    df_census_raw['state' ] = df_census_raw['state' ].astype(str).apply('{:0>2}'.format)
    df_census_raw['county'] = df_census_raw['county'].astype(str).apply('{:0>3}'.format)
    df_census_raw = df_census_raw.merge(df_fips[['State FIPS', 'County FIPS', 'County Name']]
                                          , left_on = ['state', 'county']
                                          , right_on = ['State FIPS', 'County FIPS'])
    df_census_raw.drop(['State FIPS', 'County FIPS'], axis = 1, inplace = True)
    df_census_raw = df_census_raw.set_index(['state', 'county', 'County Name', 'Year']).reset_index()

print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")

In [ ]:
# view raw data
pd.set_option('display.max_columns', None)
print(df_census_raw.shape)
print(df_census_raw.Year.unique())
df_census_raw.head(3)

In [ ]:
print("Count of '-666666666'   values in dataframe: " + str((df_census_raw.values == '-666666666'  ).sum()))
print("Count of '-999999999.0' values in dataframe: " + str((df_census_raw.values == '-999999999.0').sum()))
print("Count of 'null'         values in dataframe: " + str((df_census_raw.values == 'null'        ).sum()))

print("Count of NaN by columns: ")
print( df_census_raw.isna().sum())


# # Function to save log to a file
# def save_log_to_file():
#     with open(file_path, 'w') as f:
#         f.write(f"Matching method: {scorer.__name__}\n")
#         f.write(f"Fuzzy Match Threshold: {FUZZY_MATCH_THRESHOLD}\n")
#         f.write(f"Year: {year}\n")
#         f.write(f"County: {county}\n")
#         f.write(f"Jurisdiction: {jurisdiction}\n")
#         f.write(f"Exact APN Matches: {exact_apn_matches}\n")
#         f.write(f"Exact Street Matches: {exact_street_matches}\n")
#         f.write(f"Fuzzy Street Matches: {fuzzy_street_matches}\n")
#         f.write(f"Manual Checks: {manual_checks}\n")
#         f.write(f"Records Processed: {records_processed}\n")
#         f.write(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")


***

## Processing

***

In [ ]:
## Make copy of data frame
df_census = df_census_raw.copy()


# Replace weird missing values with np.nan
# Melt data from wide to long
# Convert imported values to numeric
# Merge cleam label field, variable mapping, race/ethnicity, and sorting field
# Remove unneeded columns
# Manually check column names and clean as needed
# Adjust dollars for inflation, if needed
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 1...')
    df_census = acs_processing_1(df_census, df_vars, indicator_name, geography, year_end, path_main, path_git)
    display(df_census.head(3))

# Reorganize margin of error fields
# Create "Categorical" race/ethnicity field for sorting
# Sort by geography, variable mapping, and race/ethnicity
# sort and then remove categorical field
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 2...')
    df_census = acs_processing_2(df_census, geography, margin_of_error)
    display(df_census.head(3))

# Final processing step for ACS data
# Link various FIPS codes
# Roll up population/households/SE's to the desired geography and variable groupings
# Calculate percentages by geography, race/ethnicity, and variables
if sample_type in ['ACS', 'SUBJECT']:
    print('Processing 3...')
    if geography == 'Tracts':
        df_tracts1, df_tracts2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars, df_fips)
        display(df_tracts1.head(3), df_tracts2.head(3))
    if geography == 'Counties':
        df_counties1, df_counties2, df_mpo1, df_mpo2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars, df_fips)
        display(df_mpo1.head(3), df_mpo2.head(3))
    if geography == 'MSA':
        df_msa1, df_msa2 = acs_processing_3(df_census, indicator_name, geography, percentages, margin_of_error, MOE_thresh, num_vars)
        display(df_msa1.head(3), df_msa2.head(3))


# Clean missing values, standardize how the categories are assigned by number, standardize state FIPS code
# Convert weighted column to integer, convert value fields to string to use as merge field
# Reshape data dictionary of values/descriptions and reorganize columns
# Merge meaningful value descriptions onto imported data
if sample_type in ['PUMS', 'FOODSEC']:
    print('Processing 1...')
    df_census, groups = pums_processing_1(df_census, df_vars, sample_type, weight)
    print('Groups: ' + ', '.join(groups))
    display(df_census.head(3))

# Remove rows with missing values
# Only keep description mappings, remove the original PUMS values
if sample_type in ['PUMS', 'FOODSEC']:
    print('Processing 2...')
    df_census, groups = pums_processing_2(df_census, groups, indicator_name, dict_fips, path_config0, path_git)
    print('Groups: ' + ', '.join(groups))
    display(df_census.head(3))

# Rollup using suggested weight field
# Merge on PUMA name field
if sample_type == 'PUMS':
    print('Processing 3...')
    df_puma, df_counties, df_msa, df_mpo = pums_processing_3(df_census, indicator_name, weight, margin_of_error, MOE_thresh, percentages, groups)
    display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))

    

In [ ]:
# Final renaming of tables for cleanliness
if geography == 'Tracts':
    df_tracts1 = rename_census(df_tracts1        = df_tracts1
                               , geography       = geography
                               , indicator_name  = indicator_name
                               , margin_of_error = margin_of_error)
    display(df_tracts1.head(3))
if geography == 'Counties':
    df_counties1, df_mpo1 = rename_census(df_counties1      = df_counties1
                                          , df_mpo1         = df_mpo1
                                          , geography       = geography
                                          , indicator_name  = indicator_name
                                          , margin_of_error = margin_of_error)
    display(df_counties1.head(3), df_mpo1.head(3))
if geography == 'MSA':
    df_msa1 = rename_census(df_msa1           = df_msa1
                            , geography       = geography
                            , indicator_name  = indicator_name
                            , margin_of_error = margin_of_error)
    display(df_msa1.head(3))
if geography == 'PUMA':
    df_puma, df_counties, df_msa, df_mpo = rename_census(df_puma           = df_puma
                                                         , df_counties     = df_counties
                                                         , df_msa          = df_msa
                                                         , df_mpo          = df_mpo
                                                         , geography       = geography
                                                         , indicator_name  = indicator_name
                                                         , margin_of_error = margin_of_error
                                                         , groups          = groups)
    display(df_puma.head(3), df_counties.head(3), df_msa.head(3), df_mpo.head(3))

***

## Exporting

***

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )

if geography == 'Tracts':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Tracts '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_tracts1.to_excel(writer, index = False, sheet_name = 'Tracts')
        # df_tracts2.to_excel(writer, index = False, sheet_name = 'Tracts wide')

if geography == 'Counties':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_counties1.to_excel(writer, index = False, sheet_name = 'Counties')
        # df_counties2.to_excel(writer, index = False, sheet_name = 'Counties wide')
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_mpo1.to_excel(writer, index = False, sheet_name = 'MPO')
        # df_mpo2.to_excel(writer, index = False, sheet_name = 'MPO wide')
             

if geography == 'MSA':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+estimate+'.xlsx'), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+estimate+'.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_msa1.to_excel(writer, index = False, sheet_name = 'MSA')
        # df_msa2.to_excel(writer, index = False, sheet_name = 'MSA wide')


if geography == 'PUMA':
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' PUMA '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
    #     df_puma.to_excel(writer, index = False, sheet_name = 'PUMA')
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' Counties '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        df_counties.to_excel(writer, index = False, sheet_name = 'Counties')
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MSA '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
    #     df_msa.to_excel(writer, index = False, sheet_name = 'MSA')
    with pd.ExcelWriter(os.path.join(path_out_xlsx, indicator_name+' MPO '+re.sub('ACS', 'PUMS', estimate)+'.xlsx'), engine='xlsxwriter') as writer:
        df_mpo.to_excel(writer, index = False, sheet_name = 'MPO')


print('')
print("Successfully exported")